### Mount Drive, initialise Spark, and load the verified Task 2 outputs.


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json, time, os, platform
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("7006SCN_Task4_WESAD_Distributed_Computing")
    .config("spark.master", "local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

TASK2_DIR = Path("/content/drive/MyDrive/wesad_task2_outputs")
TRAIN_PATH = TASK2_DIR / "train_processed"
TEST_PATH = TASK2_DIR / "test_processed"
TASK2_MANIFEST_PATH = TASK2_DIR / "task2_manifest.json"

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Task 2 train/test outputs are missing. Complete Task 2 first.")
if not TASK2_MANIFEST_PATH.exists():
    raise FileNotFoundError("Task 2 manifest is missing.")

task2_manifest = json.loads(TASK2_MANIFEST_PATH.read_text(encoding="utf-8"))
if task2_manifest.get("status") != "verified":
    raise RuntimeError("Task 2 manifest is not marked verified.")

print("Spark version:", spark.version)
print("Spark UI URL:", spark.sparkContext.uiWebUrl)
print("Application ID:", spark.sparkContext.applicationId)
print("Task 2 source:", TASK2_DIR)
print("Spark master:", spark.sparkContext.master)


Mounted at /content/drive
Spark version: 4.0.4
Spark UI URL: http://fde2c50f0932:4040
Application ID: local-1793085405869
Task 2 source: /content/drive/MyDrive/wesad_task2_outputs
Spark master: local[*]


### Inspect effective Spark resource settings and the input partition layout.


In [2]:
conf = dict(spark.sparkContext.getConf().getAll())
keys_of_interest = [
    "spark.master",
    "spark.driver.memory",
    "spark.executor.memory",
    "spark.executor.cores",
    "spark.default.parallelism",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
]
for key in keys_of_interest:
    print(f"{key}: {conf.get(key, '(not explicitly set)')}")

# Read train and test separately so any failure is isolated to its own output.
train_full = spark.read.parquet(str(TRAIN_PATH))
test_full = spark.read.parquet(str(TEST_PATH))

required = {"features", "label", "target_text", "subject_id", "sample_index"}
for name, frame in [("train", train_full), ("test", test_full)]:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{name} data missing columns: {missing}")

print("\\nTraining rows:", f"{train_full.count():,}")
print("Training partitions:", train_full.rdd.getNumPartitions())
print("Test rows:", f"{test_full.count():,}")
print("Test partitions:", test_full.rdd.getNumPartitions())
print("Training schema:")
train_full.printSchema()


spark.master: local[*]
spark.driver.memory: (not explicitly set)
spark.executor.memory: (not explicitly set)
spark.executor.cores: (not explicitly set)
spark.default.parallelism: (not explicitly set)
spark.sql.shuffle.partitions: 8
spark.sql.adaptive.enabled: true
\nTraining rows: 25,137,704
Training partitions: 5
Test rows: 6,332,899
Test partitions: 3
Training schema:
root
 |-- features: vector (nullable = true)
 |-- label: double (nullable = true)
 |-- target_text: string (nullable = true)
 |-- subject_id: string (nullable = true)
 |-- sample_index: long (nullable = true)



### Create a deterministic bounded sample for repeatable performance experiments.


In [3]:
MAX_ROWS = 200_000
MAX_TEST_ROWS = 100_000
SEED = 42

train_count = train_full.count()
test_count = test_full.count()

def bounded_sample(frame, row_count, limit, seed):
    if row_count <= limit:
        return frame
    fraction = min(1.0, (limit * 1.25) / row_count)
    labels = [r["label"] for r in frame.select("label").distinct().collect()]
    fractions = {label: fraction for label in labels}
    sampled = frame.stat.sampleBy("label", fractions=fractions, seed=seed)
    sampled_count = sampled.count()
    if sampled_count > limit:
        sampled = sampled.orderBy(F.rand(seed=seed)).limit(limit)
    return sampled

train = bounded_sample(train_full, train_count, MAX_ROWS, SEED)
test = bounded_sample(test_full, test_count, MAX_TEST_ROWS, SEED + 1)

actual_train = train.count()
actual_test = test.count()
train = train.repartition(4, "label").cache()
test = test.repartition(4, "label").cache()

print("Full training rows:", f"{train_count:,}")
print("Training benchmark sample:", f"{actual_train:,}")
print("Full test rows:", f"{test_count:,}")
print("Test benchmark sample:", f"{actual_test:,}")
print("Training sample partitions after repartition:", train.rdd.getNumPartitions())
print("Test sample partitions after repartition:", test.rdd.getNumPartitions())
print("Training sample label counts:")
train.groupBy("label").count().orderBy("label").show()


Full training rows: 25,137,704
Training benchmark sample: 200,000
Full test rows: 6,332,899
Test benchmark sample: 100,000
Training sample partitions after repartition: 4
Test sample partitions after repartition: 4
Training sample label counts:
+-----+------+
|label| count|
+-----+------+
|  0.0|156131|
|  1.0| 43869|
+-----+------+



### Benchmark a baseline aggregation without caching or repartitioning.


In [4]:
baseline = train.select("label", "subject_id", "sample_index", "features")
baseline_partitions = baseline.rdd.getNumPartitions()

start = time.perf_counter()
baseline_result = (
    baseline.groupBy("label")
    .agg(F.count("*").alias("rows"), F.countDistinct("subject_id").alias("participants"))
    .collect()
)
baseline_seconds = time.perf_counter() - start

print("Baseline partition count:", baseline_partitions)
print(f"Baseline aggregation time: {baseline_seconds:.2f} seconds")
print("Baseline aggregation:")
for row in baseline_result:
    print(row)


Baseline partition count: 4
Baseline aggregation time: 1.18 seconds
Baseline aggregation:
Row(label=0.0, rows=156131, participants=12)
Row(label=1.0, rows=43869, participants=12)


### Repartition the experiment sample and benchmark the same aggregation.


In [5]:
target_partitions = 8
repartitioned = baseline.repartition(target_partitions, "label")

start = time.perf_counter()
repartitioned_result = (
    repartitioned.groupBy("label")
    .agg(F.count("*").alias("rows"), F.countDistinct("subject_id").alias("participants"))
    .collect()
)
repartitioned_seconds = time.perf_counter() - start

print("Original partitions:", baseline_partitions)
print("Repartitioned partitions:", repartitioned.rdd.getNumPartitions())
print(f"Repartitioned aggregation time: {repartitioned_seconds:.2f} seconds")
print("Repartitioned aggregation:")
for row in repartitioned_result:
    print(row)
print("Compare timings cautiously: runtime load, warm-up, and shuffle overhead affect results.")


Original partitions: 4
Repartitioned partitions: 8
Repartitioned aggregation time: 1.00 seconds
Repartitioned aggregation:
Row(label=1.0, rows=43869, participants=12)
Row(label=0.0, rows=156131, participants=12)
Compare timings cautiously: runtime load, warm-up, and shuffle overhead affect results.


### Persist the repeatedly used training sample and compare the repeated-action timings.


In [6]:
persisted = train.persist(StorageLevel.MEMORY_AND_DISK)

start = time.perf_counter()
persisted.count()
cache_materialisation_seconds = time.perf_counter() - start

# Warm-up action is intentionally not included in the timed repeated aggregations.
persisted.groupBy("label").count().collect()

start = time.perf_counter()
persisted.groupBy("label").count().collect()
cached_repeat_seconds = time.perf_counter() - start

# Force a fresh uncached lineage and run the identical aggregation.
uncached = train.unpersist(blocking=True)
uncached = train.repartition(4, "label")

start = time.perf_counter()
uncached.groupBy("label").count().collect()
uncached_repeat_seconds = time.perf_counter() - start

print("Storage level used:", StorageLevel.MEMORY_AND_DISK)
print(f"Cache materialisation time: {cache_materialisation_seconds:.2f} seconds")
print(f"Repeated aggregation with cache: {cached_repeat_seconds:.2f} seconds")
print(f"Aggregation without cache: {uncached_repeat_seconds:.2f} seconds")
print("A caching benefit should only be claimed if repeated comparable actions are consistently faster.")


Storage level used: Disk Memory Serialized 1x Replicated
Cache materialisation time: 0.29 seconds
Repeated aggregation with cache: 0.19 seconds
Aggregation without cache: 3.84 seconds
A caching benefit should only be claimed if repeated comparable actions are consistently faster.


### Capture Spark UI evidence and inspect the execution plan for the distributed aggregation.


In [7]:
print("Spark UI URL:", spark.sparkContext.uiWebUrl)
print("Application ID:", spark.sparkContext.applicationId)
print("Open the Spark UI during this active runtime and capture the Jobs and Stages tabs.")
print("If the UI URL is not reachable in your browser, use the Colab Spark UI link/proxy if available; do not claim a screenshot was captured.")

print("\\nExecution plan for repartitioned aggregation:")
(
    repartitioned.groupBy("label")
    .agg(F.count("*").alias("rows"))
    .explain("formatted")
)


Spark UI URL: http://fde2c50f0932:4040
Application ID: local-1793085405869
Open the Spark UI during this active runtime and capture the Jobs and Stages tabs.
If the UI URL is not reachable in your browser, use the Colab Spark UI link/proxy if available; do not claim a screenshot was captured.
\nExecution plan for repartitioned aggregation:
== Physical Plan ==
AdaptiveSparkPlan (8)
+- HashAggregate (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- TakeOrderedAndProject (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [label#1]
Batched: true
Location: InMemoryFileIndex [file:/content/drive/MyDrive/wesad_task2_outputs/train_processed]
ReadSchema: struct<label:double>

(2) Filter
Input [1]: [label#1]
Condition : UDF(label#1, rand(42))

(3) Project
Output [2]: [label#1, rand(42) AS _nondeterministic#61]
Input [1]: [label#1]

(4) TakeOrderedAndProject
Input [2]: [label#1, _nondeterministic#61]
Argum

### Summarise the timing, partitioning, persistence, and resource configuration evidence.


In [8]:
RESULTS_DIR = Path("/content/drive/MyDrive/wesad_task4_outputs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

conf = dict(spark.sparkContext.getConf().getAll())
benchmark = {
    "spark_version": spark.version,
    "application_id": spark.sparkContext.applicationId,
    "spark_ui_url": spark.sparkContext.uiWebUrl,
    "spark_master": spark.sparkContext.master,
    "driver_memory": conf.get("spark.driver.memory", "Spark default; not explicitly set"),
    "executor_memory": conf.get("spark.executor.memory", "Spark default; not explicitly set"),
    "executor_cores": conf.get("spark.executor.cores", "Local mode; executor cores not separately configured"),
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "adaptive_execution": spark.conf.get("spark.sql.adaptive.enabled"),
    "full_train_rows": int(train_count),
    "sample_train_rows": int(actual_train),
    "full_test_rows": int(test_count),
    "sample_test_rows": int(actual_test),
    "baseline_partitions": int(baseline_partitions),
    "repartitioned_partitions": int(repartitioned.rdd.getNumPartitions()),
    "target_repartition_count": int(target_partitions),
    "baseline_aggregation_seconds": round(baseline_seconds, 4),
    "repartitioned_aggregation_seconds": round(repartitioned_seconds, 4),
    "cache_materialisation_seconds": round(cache_materialisation_seconds, 4),
    "cached_repeat_aggregation_seconds": round(cached_repeat_seconds, 4),
    "uncached_repeat_aggregation_seconds": round(uncached_repeat_seconds, 4),
    "benchmark_scope": "bounded deterministic sample; not a full-dataset performance claim",
    "spark_ui_screenshot_required": True,
    "status": "completed"
}
(RESULTS_DIR / "task4_benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")

print("Saved benchmark:", RESULTS_DIR / "task4_benchmark.json")
print(json.dumps(benchmark, indent=2))
print("Output directory:", RESULTS_DIR)


Saved benchmark: /content/drive/MyDrive/wesad_task4_outputs/task4_benchmark.json
{
  "spark_version": "4.0.4",
  "application_id": "local-1793085405869",
  "spark_ui_url": "http://fde2c50f0932:4040",
  "spark_master": "local[*]",
  "driver_memory": "Spark default; not explicitly set",
  "executor_memory": "Spark default; not explicitly set",
  "executor_cores": "Local mode; executor cores not separately configured",
  "default_parallelism": 2,
  "shuffle_partitions": "8",
  "adaptive_execution": "true",
  "full_train_rows": 25137704,
  "sample_train_rows": 200000,
  "full_test_rows": 6332899,
  "sample_test_rows": 100000,
  "baseline_partitions": 4,
  "repartitioned_partitions": 8,
  "target_repartition_count": 8,
  "baseline_aggregation_seconds": 1.1848,
  "repartitioned_aggregation_seconds": 1.0025,
  "cache_materialisation_seconds": 0.2917,
  "cached_repeat_aggregation_seconds": 0.1917,
  "uncached_repeat_aggregation_seconds": 3.8381,
  "benchmark_scope": "bounded deterministic samp

In [48]:

import socket, threading, requests
from bs4 import BeautifulSoup
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from html import escape
from google.colab import output

SPARK = "http://localhost:4040"

def get_table(path, rows=6):
    try:
        r = requests.get(SPARK + path, timeout=12)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        table = soup.find("table")
        if not table:
            return "<p>Table unavailable.</p>"

        for cell in table.find_all(["td", "th"]):
            if cell.get_text(" ", strip=True).lower() == "nan":
                cell.clear()
                cell.string = "—"

        table_rows = table.find_all("tr")
        clean = BeautifulSoup("<table></table>", "html.parser").table
        for tr in table_rows[:rows + 1]:
            clean.append(tr)
        return str(clean)
    except Exception as e:
        return "<p>Could not retrieve section: " + escape(str(e)) + "</p>"

conf = dict(spark.sparkContext.getConf().getAll())
settings = [
    ("Spark version", spark.version),
    ("Application ID", spark.sparkContext.applicationId),
    ("Execution mode", spark.sparkContext.master),
    ("Default parallelism", str(spark.sparkContext.defaultParallelism)),
    ("Shuffle partitions", spark.conf.get("spark.sql.shuffle.partitions")),
    ("Adaptive execution", spark.conf.get("spark.sql.adaptive.enabled")),
    ("Executor memory", conf.get("spark.executor.memory", "Not explicitly set")),
    ("Executor cores", conf.get("spark.executor.cores", "Not explicitly set")),
]
settings_rows = "".join(
    f"<tr><td>{escape(k)}</td><td>{escape(v)}</td></tr>"
    for k, v in settings
)

page = """
<!doctype html><html><head><meta charset="utf-8">
<title>Task 4 — Distributed Computing Evidence</title>
<style>
body{font-family:Arial,sans-serif;max-width:700px;margin:16px auto;
color:#16345a;font-size:10px;background:white}
h1{font-size:15px;margin-bottom:5px}
.subtitle{font-size:9px;color:#526782}
h2{font-size:11px;font-weight:600;margin-top:16px;
padding-bottom:5px;border-bottom:1px solid #cbd2d9}
table{border-collapse:collapse;width:100%;font-size:9px;
color:#183451;table-layout:auto}
th,td{border:1px solid #d0d2d5;padding:4px;text-align:left;
overflow-wrap:anywhere;background:white}
th{background:#e8e8e8;font-weight:600}
.note{padding:6px;font-size:9px;border:1px solid #d0d2d5}
</style></head><body>
<h1>Task 4 — Distributed Computing Evidence</h1>
<div class="subtitle">WESAD | Spark execution, configuration and optimisation</div>
<h2>Runtime and Resource Configuration</h2>
<table><tr><th>Configuration</th><th>Observed value</th></tr>
""" + settings_rows + """
</table>
<div class="note">Settings are reported as observed. Local mode does not demonstrate a multi-node cluster.</div>
<h2>Jobs — Execution Evidence</h2>
""" + get_table("/jobs/", 6) + """
<h2>Stages — Task and Stage Evidence</h2>
""" + get_table("/stages/", 5) + """
</body></html>
"""

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        data = page.encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)
    def log_message(self, *args):
        pass

with socket.socket() as s:
    s.bind(("0.0.0.0", 0))
    port = s.getsockname()[1]

server = ThreadingHTTPServer(("0.0.0.0", port), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()

output.serve_kernel_port_as_window(port, path="/")


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>